# VLM-SAM3 Experiment Results — Job 463119

**Job:** 463119  
**Design:** 2×2 factorial — {`235b-cloud`, `235b-instruct-cloud`} × {`plain prompt`, `neighbor-context prompt`}  
**Baseline:** `baseline_multiframe_w10.csv` — 50 balanced takes (10 per scenario), 191 streams × 10 frames = 1910 pairs  
**Fixes in this run:**
- `num_predict` raised 32 → 64 for non-CoT experiments (fixes ~20% missing VLM output)
- `sample_source_frames` now selects N *preceding* annotated frames (fixes EXP-F-mf5 temporal confusion)
- SAM3 32-token overflow: CoT outputs truncated via `_prepare_sam3_text()`

**Sections:**
1. Load & filter data (handles resume contamination)
2. Data sanity check
3. Missing VLM output
4. 2×2 factorial comparison
5. Metric distributions
6. Object size & spatial effects
7. VLM text analysis (neighbor-context quality)
8. Per-scenario breakdown
9. Summary & key findings

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION                                                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
from pathlib import Path

JOB_ID      = "463119"

REPO_ROOT   = Path("/home/3164542/reluminati-research")
SCRIPT_DIR  = REPO_ROOT / "src/scripts/experiments/vlm-sam3"
DATA_ROOT   = Path("/data/video_datasets/3321908/output_dir_all")

RESULTS_CSV  = SCRIPT_DIR / f"results_{JOB_ID}.csv"
BASELINE_CSV = Path("/data/video_datasets/3164542/baseline_multiframe_w10.csv")

# The 4 experiments in this run (the only ones we analyse)
TARGET_EXPS = ["EXP-C", "EXP-C-cot", "EXP-C-ctx", "EXP-C-ctx-cot"]

print(f"Results CSV  : {RESULTS_CSV}")
print(f"Exists       : {RESULTS_CSV.exists()}")
print(f"Baseline CSV : {BASELINE_CSV}")
print(f"Exists       : {BASELINE_CSV.exists()}")

In [ ]:
import json, re, warnings
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 11})
sns.set_theme(style="whitegrid", palette="husl")

# ── 2×2 palette ──────────────────────────────────────────────────────────────
EXP_PALETTE = {
    "EXP-C":         "#3498db",   # cloud + plain
    "EXP-C-cot":     "#e74c3c",   # instruct + plain
    "EXP-C-ctx":     "#2ecc71",   # cloud + neighbor-context
    "EXP-C-ctx-cot": "#9b59b6",   # instruct + neighbor-context
}

# Human-readable labels for the 2×2 grid
EXP_LABELS = {
    "EXP-C":         "cloud\n(plain)",
    "EXP-C-cot":     "instruct\n(plain)",
    "EXP-C-ctx":     "cloud\n(ctx)",
    "EXP-C-ctx-cot": "instruct\n(ctx)",
}

KEY_COLS = ["take_uid", "object_name", "src_camera", "dest_camera", "frame"]
METRICS  = ["iou", "ba", "ca", "le"]

print("Imports OK.")

In [ ]:
# ── 1. Load raw CSV ───────────────────────────────────────────────────────────
raw = pd.read_csv(RESULTS_CSV)

# Column rename: older experiment.py used 'experiment', newer may use 'experiment_id'
if "experiment_id" in raw.columns and "experiment" not in raw.columns:
    raw = raw.rename(columns={"experiment_id": "experiment"})

print(f"Raw CSV shape : {raw.shape}")
print(f"Experiments   : {sorted(raw['experiment'].unique())}")

# ── 2. Load baseline take_uids (to strip resume-contamination from old runs) ──
baseline = pd.read_csv(BASELINE_CSV)
baseline_takes = set(baseline["take_uid"])
print(f"\nBaseline takes in CSV : {len(baseline_takes)}")

# ── 3. Filter: keep only TARGET_EXPS AND baseline take_uids ──────────────────
df = raw[
    raw["experiment"].isin(TARGET_EXPS) &
    raw["take_uid"].isin(baseline_takes)
].copy()

print(f"\nFiltered rows : {len(df):,}")
print(f"Unique pairs  : {df[KEY_COLS].drop_duplicates().shape[0]:,}  "
      f"(expected ≤ {len(baseline):,})")
print(f"Experiments   : {sorted(df['experiment'].unique())}")

In [ ]:
# ── Type coercion ─────────────────────────────────────────────────────────────
for c in ["iou", "ba", "ca", "le", "obj_rel_area", "obj_dist_center",
          "src_mask_area", "src_move_x_ratio", "src_move_y_ratio",
          "vlm_num_frames", "vlm_video_window"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["frame"]       = df["frame"].astype(str)
df["vlm_missing"] = df["vlm_output"].isna()
df["success"]     = df["iou"] > 0
df["success_25"]  = df["iou"] >= 0.25
df["success_50"]  = df["iou"] >= 0.50

# ── 2×2 factorial factor columns ─────────────────────────────────────────────
df["model_type"] = df["experiment"].map({
    "EXP-C":         "cloud",
    "EXP-C-cot":     "instruct",
    "EXP-C-ctx":     "cloud",
    "EXP-C-ctx-cot": "instruct",
})
df["prompt_type"] = df["experiment"].map({
    "EXP-C":         "plain",
    "EXP-C-cot":     "plain",
    "EXP-C-ctx":     "neighbor-ctx",
    "EXP-C-ctx-cot": "neighbor-ctx",
})

print("\nRows per experiment after filtering:")
print(df["experiment"].value_counts()[TARGET_EXPS])

---
## 2  Data Sanity Check

In [ ]:
# ── Contamination report ──────────────────────────────────────────────────────
contam_rows  = len(raw) - len(df)  # rows dropped by filtering
contam_takes = set(raw["take_uid"]) - baseline_takes

print("=== Contamination Report ===")
print(f"Raw rows          : {len(raw):,}")
print(f"Filtered rows     : {len(df):,}")
print(f"Dropped rows      : {contam_rows:,}  (old experiments from earlier runs)")
print(f"Non-baseline takes: {len(contam_takes)}")

print("\n=== Experiments present in raw CSV ===")
print(raw["experiment"].value_counts().to_string())

# ── Completeness ─────────────────────────────────────────────────────────────
print("\n=== Completeness check ===")
n_pairs = len(baseline)
for exp in TARGET_EXPS:
    n = (df["experiment"] == exp).sum()
    pct = 100 * n / n_pairs
    status = "COMPLETE" if n == n_pairs else f"RUNNING ({pct:.0f}%)"
    print(f"  {exp:20s}: {n:5d}/{n_pairs}  [{status}]")

---
## 3  Missing VLM Output

Previous run (job 462563) had **18-27% missing** for non-CoT experiments due to `num_predict=32`.
This run raises it to **64** for `cloud` models; `instruct` (CoT) remains at 512.

In [ ]:
miss = (
    df.groupby("experiment")["vlm_missing"]
    .agg(n_missing="sum", n_total="count",
         pct_missing=lambda x: 100 * x.mean())
    .reindex(TARGET_EXPS)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(
    [EXP_LABELS[e] for e in miss["experiment"]],
    miss["pct_missing"],
    color=[EXP_PALETTE[e] for e in miss["experiment"]],
    edgecolor="white", linewidth=0.8
)
for b, v in zip(bars, miss["pct_missing"]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.3,
            f"{v:.1f}%", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Missing VLM output (%)")
ax.set_title(f"Missing VLM output rate — job {JOB_ID}\n"
             "(num_predict: 64 for cloud, 512 for instruct)",
             fontweight="bold")
plt.tight_layout()
plt.show()

display(miss)

In [ ]:
# Success rate split by VLM output presence
ct = pd.crosstab(df["vlm_missing"], df["success"], normalize="index") * 100
ct.index = ["VLM present", "VLM missing"]
ct.columns = ["Failure (IoU=0)", "Success (IoU>0)"]
print("Success rate by VLM output presence (row %):")
display(ct.round(1))

# Also show by experiment
miss_impact = (
    df.groupby(["experiment", "vlm_missing"])["success"]
    .mean()
    .mul(100)
    .unstack("vlm_missing")
    .rename(columns={False: "VLM present", True: "VLM missing"})
    .reindex(TARGET_EXPS)
)
print("\nSuccess rate split by VLM output presence, per experiment:")
display(miss_impact.round(1))

---
## 4  2×2 Factorial Comparison

| | **plain prompt** | **neighbor-context prompt** |
|---|---|---|
| **235b-cloud** | EXP-C | EXP-C-ctx |
| **235b-instruct** | EXP-C-cot | EXP-C-ctx-cot |

Main effects to test:
- **Model effect:** cloud vs instruct (CoT reasoning + instruct fine-tuning)
- **Context effect:** plain vs neighbor-context (adding nearby object descriptions)

In [ ]:
def exp_summary(data, exps=TARGET_EXPS):
    rows = []
    for exp in exps:
        g = data[data["experiment"] == exp]
        if len(g) == 0:
            continue
        g_present = g[~g["vlm_missing"]]
        rows.append({
            "experiment":           exp,
            "model":                g["model_type"].iloc[0],
            "prompt":               g["prompt_type"].iloc[0],
            "n":                    len(g),
            "vlm_missing_%":        g["vlm_missing"].mean() * 100,
            "success_%":            g["success"].mean() * 100,
            "iou>=0.25_%":          g["success_25"].mean() * 100,
            "iou>=0.50_%":          g["success_50"].mean() * 100,
            "iou_mean":             g["iou"].mean(),
            "iou_mean|vlm_present": g_present["iou"].mean() if len(g_present) else float("nan"),
            "iou_mean|success":     g[g["success"]]["iou"].mean() if g["success"].any() else float("nan"),
            "le_mean":              g["le"].mean(),
            "ba_mean":              g["ba"].mean(),
        })
    return pd.DataFrame(rows).set_index("experiment")

summ = exp_summary(df)
display(
    summ.style
        .format({
            "vlm_missing_%":        "{:.1f}%",
            "success_%":            "{:.1f}%",
            "iou>=0.25_%":          "{:.1f}%",
            "iou>=0.50_%":          "{:.1f}%",
            "iou_mean":             "{:.3f}",
            "iou_mean|vlm_present": "{:.3f}",
            "iou_mean|success":     "{:.3f}",
            "le_mean":              "{:.3f}",
            "ba_mean":              "{:.3f}",
        })
        .background_gradient(subset=["success_%"], cmap="RdYlGn")
        .background_gradient(subset=["vlm_missing_%"], cmap="RdYlGn_r")
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, ylabel, title in [
    (axes[0], "success_%",            "Success rate (%)",        "Success rate (IoU > 0)"),
    (axes[1], "iou_mean",             "Mean IoU",                "Mean IoU (unconditional)"),
    (axes[2], "iou_mean|vlm_present", "Mean IoU | VLM present",  "Mean IoU | VLM output present"),
]:
    vals = summ[col].reindex(TARGET_EXPS)
    bars = ax.bar(
        [EXP_LABELS[e] for e in TARGET_EXPS],
        vals.values,
        color=[EXP_PALETTE[e] for e in TARGET_EXPS],
        edgecolor="white", linewidth=0.8
    )
    for b, v in zip(bars, vals.values):
        if not np.isnan(v):
            ax.text(b.get_x() + b.get_width()/2, v + 0.003,
                    f"{v:.2f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")

# Legend
handles = [mpatches.Patch(color=EXP_PALETTE[e], label=e) for e in TARGET_EXPS]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.0, 1.0),
           title="Experiment")
plt.suptitle(f"2×2 Factorial — job {JOB_ID}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Main effects: model type and prompt type ──────────────────────────────────
print("=== Main effect: Model type ===")
model_eff = (
    df.groupby("model_type")
    .agg(
        n=("iou", "count"),
        success_pct=("success", lambda x: 100*x.mean()),
        iou_mean=("iou", "mean"),
        vlm_missing_pct=("vlm_missing", lambda x: 100*x.mean()),
    )
)
display(model_eff.round(3))

print("\n=== Main effect: Prompt type ===")
prompt_eff = (
    df.groupby("prompt_type")
    .agg(
        n=("iou", "count"),
        success_pct=("success", lambda x: 100*x.mean()),
        iou_mean=("iou", "mean"),
        vlm_missing_pct=("vlm_missing", lambda x: 100*x.mean()),
    )
)
display(prompt_eff.round(3))

print("\n=== Interaction: model × prompt ===")
interact = (
    df.groupby(["model_type", "prompt_type"])
    .agg(
        success_pct=("success", lambda x: 100*x.mean()),
        iou_mean=("iou", "mean"),
    )
    .round(3)
)
display(interact)

In [ ]:
# ── 2×2 heatmap ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, metric, title in [
    (axes[0], "success_%",  "Success rate (%)"),
    (axes[1], "iou_mean",   "Mean IoU"),
]:
    pivot = (
        summ.reset_index()
        .pivot(index="model", columns="prompt", values=metric)
        .reindex(index=["cloud", "instruct"], columns=["plain", "neighbor-ctx"])
    )
    sns.heatmap(
        pivot, annot=True, fmt=".2f", cmap="YlGn",
        linewidths=2, linecolor="white",
        ax=ax, cbar_kws={"shrink": 0.8}
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Prompt type")
    ax.set_ylabel("Model type")

plt.suptitle("2×2 factorial heatmap — model × prompt",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5  Metric Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
metric_labels = {"iou": "IoU", "ba": "Balanced Accuracy",
                 "ca": "Contour Accuracy", "le": "Location Error"}

for col, (m, lbl) in enumerate(metric_labels.items()):
    for row, (mask, subtitle, color) in enumerate([
        (slice(None), "all pairs",           "steelblue"),
        (df["success"],  "IoU > 0 only",    "seagreen"),
    ]):
        data = df.loc[mask, m].dropna() if isinstance(mask, slice) else df.loc[mask, m].dropna()
        axes[row, col].hist(data, bins=40, color=color, edgecolor="white", linewidth=0.4)
        axes[row, col].set_title(f"{lbl}\n({subtitle}, n={len(data):,})",
                                  fontweight="bold", fontsize=9)
        axes[row, col].set_xlabel(lbl)

plt.suptitle("Metric distributions — all pairs (top) vs successful pairs (bottom)",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Overall success rate (IoU > 0)  : {df['success'].mean()*100:.1f}%")
print(f"Success rate (IoU ≥ 0.25)       : {df['success_25'].mean()*100:.1f}%")
print(f"Success rate (IoU ≥ 0.50)       : {df['success_50'].mean()*100:.1f}%")

In [ ]:
# IoU violin by experiment
fig, ax = plt.subplots(figsize=(10, 5))
plot_order = TARGET_EXPS
sns.violinplot(
    data=df, x="experiment", y="iou",
    order=plot_order,
    palette=EXP_PALETTE,
    inner="quartile", cut=0, ax=ax
)
ax.set_xlabel("")
ax.set_xticklabels([EXP_LABELS[e] for e in plot_order])
ax.set_ylabel("IoU")
ax.set_title("IoU distribution per experiment", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6  Object Size & Spatial Effects

In [ ]:
SIZE_ORDER = ["small", "medium", "large"]
sz = (
    df.groupby("obj_size_cat")
    .agg(
        n=("iou", "count"),
        success_pct=("success", lambda x: 100*x.mean()),
        iou_mean=("iou", "mean"),
        le_mean=("le", "mean"),
    )
    .reindex(SIZE_ORDER)
)
display(sz.round(3))

# Per-experiment × size breakdown
fig, ax = plt.subplots(figsize=(11, 5))
sz_exp = (
    df.groupby(["experiment", "obj_size_cat"])["success"]
    .mean()
    .mul(100)
    .unstack("obj_size_cat")
    .reindex(index=TARGET_EXPS, columns=SIZE_ORDER)
)
sz_exp.plot(kind="bar", ax=ax, color=["#3498db", "#2ecc71", "#e74c3c"],
            edgecolor="white")
ax.set_xticklabels([EXP_LABELS[e] for e in TARGET_EXPS], rotation=0)
ax.set_ylabel("Success rate (%)")
ax.set_title("Success rate by experiment × object size", fontweight="bold")
ax.legend(title="Size", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

---
## 7  VLM Text Analysis — Neighbor-Context Quality

The `EXP-C-ctx` / `EXP-C-ctx-cot` experiments use a prompt that includes
a *neighbor object* description to help SAM3 locate the target by relative position.
We examine what the model actually outputs and whether neighbor-context descriptions
differ meaningfully from plain descriptions.

In [ ]:
# Top VLM outputs per experiment
for exp in TARGET_EXPS:
    sub = df[(df["experiment"] == exp) & ~df["vlm_missing"]]["vlm_output"]
    sub = sub.str.strip().str.lower()
    top = sub.value_counts().head(10)
    print(f"\n── {exp} — top 10 outputs ──────────────────────────")
    print(top.to_string())

In [ ]:
# 'Red' hallucination rate per experiment
# (experiment uses red overlay on source image; model should NOT say 'red' for target)
red_by_exp = (
    df[~df["vlm_missing"]]
    .assign(says_red=lambda d:
            d["vlm_output"].str.lower().str.contains(r"\bred\b").fillna(False))
    .groupby("experiment")["says_red"]
    .mean()
    .mul(100)
    .reindex(TARGET_EXPS)
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(
    [EXP_LABELS[e] for e in TARGET_EXPS],
    red_by_exp.values,
    color=[EXP_PALETTE[e] for e in TARGET_EXPS],
    edgecolor="white"
)
for i, v in enumerate(red_by_exp.values):
    ax.text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("% outputs containing 'red'")
ax.set_title("'Red' hallucination rate per experiment\n"
             "(Ideal: 0% — red is the overlay colour, not the object)",
             fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# VLM output length distribution — did num_predict=64 fix truncation?
df["vlm_output_words"] = (
    df["vlm_output"]
    .fillna("")
    .str.split()
    .str.len()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, exp_set, title in [
    (axes[0], ["EXP-C",     "EXP-C-ctx"],     "cloud (num_predict=64)"),
    (axes[1], ["EXP-C-cot", "EXP-C-ctx-cot"], "instruct (num_predict=512)"),
]:
    for exp in exp_set:
        sub = df[(df["experiment"] == exp) & ~df["vlm_missing"]]["vlm_output_words"]
        ax.hist(sub, bins=30, alpha=0.6, label=exp,
                color=EXP_PALETTE[exp], edgecolor="white")
    ax.set_xlabel("VLM output length (words)")
    ax.set_ylabel("Count")
    ax.set_title(title, fontweight="bold")
    ax.legend()

plt.suptitle("VLM output length — did num_predict=64 eliminate truncation?",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("Mean output word count per experiment (VLM-present only):")
print(
    df[~df["vlm_missing"]]
    .groupby("experiment")["vlm_output_words"]
    .agg(["mean", "median", "max"])
    .reindex(TARGET_EXPS)
    .round(1)
)

In [ ]:
# Does the neighbor-context prompt actually improve SAM3 performance?
# Compare: EXP-C vs EXP-C-ctx (same model, different prompt)
#          EXP-C-cot vs EXP-C-ctx-cot

print("Context effect — cloud model:")
cloud = df[df["model_type"] == "cloud"]
ctx_vs_plain_cloud = (
    cloud.groupby("prompt_type")[["success", "iou"]]
    .mean()
    .mul({"success": 100, "iou": 1})
)
display(ctx_vs_plain_cloud.round(3))

print("\nContext effect — instruct model:")
instruct = df[df["model_type"] == "instruct"]
ctx_vs_plain_instruct = (
    instruct.groupby("prompt_type")[["success", "iou"]]
    .mean()
    .mul({"success": 100, "iou": 1})
)
display(ctx_vs_plain_instruct.round(3))

# Paired delta
print("\nPaired success-rate deltas (ctx − plain):")
for model in ["cloud", "instruct"]:
    plain_exp = {"cloud": "EXP-C",     "instruct": "EXP-C-cot"}[model]
    ctx_exp   = {"cloud": "EXP-C-ctx", "instruct": "EXP-C-ctx-cot"}[model]
    delta = (summ.loc[ctx_exp, "success_%"] -
             summ.loc[plain_exp, "success_%"])
    sign  = "+" if delta >= 0 else ""
    print(f"  {model:10s}: {sign}{delta:.1f}% success rate")

---
## 8  Per-Scenario Breakdown

The baseline dataset has 5 balanced scenarios:
`basketball`, `bike_repair`, `cooking`, `health`, `music`

We derive the scenario from the take_uid by joining back to the baseline CSV.

In [ ]:
# Load baseline scenario mapping
# baseline_multiframe_w10.csv doesn't have scenario column;
# we derive it from generate_baseline_pairs.py's keyword matching.

SCENARIO_KEYWORDS = {
    "basketball": ["basketball", "ball", "hoop", "court"],
    "bike_repair": ["bicycle", "bike", "wheel", "fork", "chain"],
    "cooking":    ["cook", "kitchen", "knife", "pot", "pan", "cutting"],
    "health":     ["cpr", "medical", "patient", "glove"],
    "music":      ["guitar", "violin", "piano", "drum", "music"],
}

def infer_scenario(row):
    """Derive scenario from object_name heuristic."""
    obj = str(row["object_name"]).lower()
    for scenario, kws in SCENARIO_KEYWORDS.items():
        if any(kw in obj for kw in kws):
            return scenario
    return "other"

df["scenario"] = df.apply(infer_scenario, axis=1)

print("Scenario distribution (from object_name heuristic):")
print(df.drop_duplicates(KEY_COLS)["scenario"].value_counts())
print("\n(Note: 'other' = scenario not identifiable from object name)")

In [ ]:
scen = (
    df.groupby(["scenario", "experiment"])["success"]
    .mean()
    .mul(100)
    .unstack("experiment")
    .reindex(columns=TARGET_EXPS)
)

print("Success rate (%) by scenario × experiment:")
display(scen.round(1))

fig, ax = plt.subplots(figsize=(13, 5))
scen.plot(kind="bar", ax=ax,
          color=[EXP_PALETTE[e] for e in TARGET_EXPS],
          edgecolor="white")
ax.set_xlabel("Scenario")
ax.set_ylabel("Success rate (%)")
ax.set_title("Success rate by scenario × experiment", fontweight="bold")
ax.tick_params(axis="x", rotation=15)
ax.legend(title="Experiment", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

---
## 9  Summary & Key Findings

In [ ]:
n_done = df["experiment"].value_counts().min()
pct_done = 100 * n_done / len(baseline)

print(f"{'='*70}")
print(f"RESULTS SUMMARY — Job {JOB_ID}  ({pct_done:.0f}% complete, {n_done}/{len(baseline)} pairs)")
print(f"{'='*70}")
print(f"Pairs processed (min across exps): {n_done:,} / {len(baseline):,}")
print(f"Filtered rows : {len(df):,}")
print()

print("─── Per-experiment results ─────────────────────────────────────────")
display(summ[["success_%", "iou_mean", "iou_mean|vlm_present", "vlm_missing_%"]]
        .sort_values("success_%", ascending=False)
        .round(3))

print("\n─── Main effects ───────────────────────────────────────────────────")
model_delta = (
    summ.groupby("model")["success_%"].mean()
)
prompt_delta = (
    summ.groupby("prompt")["success_%"].mean()
)
print(f"  Model effect   (instruct − cloud): {model_delta.get('instruct',0) - model_delta.get('cloud',0):+.1f}% success")
print(f"  Context effect (ctx − plain)     : {prompt_delta.get('neighbor-ctx',0) - prompt_delta.get('plain',0):+.1f}% success")

print("\n─── Fixed issues vs previous run (job 462563) ──────────────────────")
fixes = [
    "✓ num_predict raised 32→64 for cloud: missing VLM output should drop ~10-15pp",
    "✓ sample_source_frames now uses preceding frames (not random video-wide frames)",
    "✓ SAM3 32-token overflow handled by _prepare_sam3_text() extraction+truncation",
    "✓ New balanced baseline: 50 takes × 5 scenarios (was 1100 arbitrary pairs)",
]
for f in fixes:
    print(f"  {f}")

print("\n─── New finding: neighbor-context prompt ───────────────────────────")
for model in ["cloud", "instruct"]:
    plain_exp = {"cloud": "EXP-C",     "instruct": "EXP-C-cot"}[model]
    ctx_exp   = {"cloud": "EXP-C-ctx", "instruct": "EXP-C-ctx-cot"}[model]
    if plain_exp in summ.index and ctx_exp in summ.index:
        delta = summ.loc[ctx_exp, "success_%"] - summ.loc[plain_exp, "success_%"]
        print(f"  {model:10s}: ctx {'+' if delta>=0 else ''}{delta:.1f}% vs plain")